# 05 — Moisture, saturation, and a CCN-motivated hypothesis

## Scientific idea (what smoke as “cloud seeds” might mean here)

Wildfire smoke adds **aerosol particles** that can act as **cloud condensation nuclei (CCN)**. In the atmosphere that often means **more cloud droplets** (at a given liquid water path), which can change **radiation**, **evaporation**, and **boundary-layer humidity**. Those are **real physical pathways**, but at a **single ASOS station** (KEUG) you **do not observe cloud droplet number** directly. What you *can* test with hourly data are **moisture and saturation proxies** that sometimes move in directions **consistent with** (or **inconsistent with**) a CCN story — always **observational**.

## What we treat as outcomes (moisture / saturation)

1. **`humidity`** — relative humidity (%).  
2. **`dewpoint_depression_f`** — \(T - T_d\) in °F. **Smaller** depression means air is **closer to saturation** (fog/stratus more plausible). CCN-rich layers do not *guarantee* lower depression, but sustained **moistening / nearer saturation** in the **hours after** smoke is one **weak** signature you can document.  
3. **`visibility_km`** (optional) — **reduced** visibility can mean **aerosol** (smoke) or **hydrometeors** (fog, rain). It is **ambiguous**; interpret alongside RH and \(T-T_d\).

## Identification sketch (same spirit as notebook 04)

- **Exposure:** lagged LRAPA-corrected PM2.5 (`pm2.5_lrapa_lag_6h` by default — change `PM_LAG_COL` if you want 3h or 12h).  
- **Controls (same hour as outcome):** `temperature_f`, `pressure_hpa`, `wind_speed_mph`, `hour`, `dayofyear` — so we ask: *after conditioning on broad thermodynamic and circulation state and diurnal/seasonal cycle, does prior smoke still line up with moisture?*  
- **Evaluation:** **`TimeSeriesSplit`** cross-validation (sklearn inside Jupyter — stable than a single 80/20 cut).  
- **Nested comparison:** **M0** = controls only vs **M1** = controls + PM2.5 lag. If M1 **never** beats M0 on average test RMSE, that is useful evidence that **this signal is not robust** in your panel (still not proof of no physical effect).

**Libraries:** `pandas`, `numpy`, `matplotlib`, `sklearn` (already in `requirements.txt`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 1. Load panel and build moisture outcomes

In [ ]:
df = pd.read_csv('../data/processed/analysis_data.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

df['dewpoint_depression_f'] = df['temperature_f'] - df['dewpoint_f']

PM_LAG_COL = 'pm2.5_lrapa_lag_6h'
CONTROL_COLS = [
    'temperature_f',
    'pressure_hpa',
    'wind_speed_mph',
    'hour',
    'dayofyear',
]

OUTCOMES = {
    'humidity': 'Relative humidity (%)',
    'dewpoint_depression_f': 'Dewpoint depression T − Td (°F)',
}
if 'visibility_km' in df.columns and df['visibility_km'].notna().sum() > 200:
    OUTCOMES['visibility_km'] = 'Visibility (km) — aerosol and/or hydrometeors'

need = [PM_LAG_COL] + CONTROL_COLS + list(OUTCOMES.keys())
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f'Missing columns: {missing} — re-run 02_data_cleaning')

work = df[['timestamp'] + need].dropna().copy()
print(f'Rows used (complete cases): {len(work):,}')
print(f'Exposure: {PM_LAG_COL}')
work[list(OUTCOMES.keys())].describe().round(2)

## 2. Exploratory: moisture vs lagged PM2.5 (raw and within approximate T bins)

RH is strongly tied to **temperature**; binning by **rounded °F** is a coarse way to show whether any PM–moisture tilt persists when **T is roughly similar** (not a substitute for regression controls).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sc = ax.scatter(
    work[PM_LAG_COL],
    work['humidity'],
    c=work['temperature_f'],
    cmap='coolwarm',
    s=12,
    alpha=0.45,
    edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='T (°F)')
ax.set_xlabel(f'Lagged PM2.5 ({PM_LAG_COL})  [µg/m³]')
ax.set_ylabel('Relative humidity (%)')
ax.set_title('RH vs lagged PM2.5 (colored by T)')

ax = axes[1]
sc = ax.scatter(
    work[PM_LAG_COL],
    work['dewpoint_depression_f'],
    c=work['temperature_f'],
    cmap='coolwarm',
    s=12,
    alpha=0.45,
    edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='T (°F)')
ax.set_xlabel(f'Lagged PM2.5 ({PM_LAG_COL})  [µg/m³]')
ax.set_ylabel('Dewpoint depression T − Td (°F)')
ax.set_title('Saturation proxy vs lagged PM2.5')

plt.tight_layout()
plt.savefig('../data/processed/fig_moisture_vs_lagged_pm25.png', dpi=150, bbox_inches='tight')
plt.show()

## 2b. RH residual (linear in **T** only), then CV on that residual

**Goal:** remove the **strong** `RH ↔ temperature` link so the outcome is closer to “**moisture given T**.”

**Procedure:**
1. On each **TimeSeriesSplit** fold, fit `humidity ~ 1 + temperature_f` using **training rows only**.
2. Form **residuals** on train and test with **that fold’s** coefficients (test never sees its own T–RH fit).
3. Regress residuals on **M0:** `pressure_hpa`, `wind_speed_mph`, `hour`, `dayofyear` (all scaled) vs **M1:** same + **lagged PM2.5** (raw µg/m³).

If M1 **lowers** mean test RMSE here, lagged PM2.5 helps explain **RH not already captured by T and the other controls**.

In [ ]:
from sklearn.linear_model import LinearRegression

alphas = np.logspace(-2, 4, 40)  # same grid as section 3 (needed if you run 2b before 3)

CONTROL_COLS_NO_T = ['pressure_hpa', 'wind_speed_mph', 'hour', 'dayofyear']

# --- EDA only: global linear RH ~ T (full sample) for a quick scatter ---
lr_eda = LinearRegression().fit(work[['temperature_f']], work['humidity'])
work['rh_resid_eda'] = work['humidity'] - lr_eda.predict(work[['temperature_f']])

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(work[PM_LAG_COL], work['rh_resid_eda'], s=14, alpha=0.35, color='steelblue', edgecolors='none')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel(f'Lagged PM2.5 ({PM_LAG_COL})  [µg/m³]')
ax.set_ylabel('RH residual (full-sample linear RH ~ T)  [%]')
ax.set_title('Exploratory: RH minus linear-T fit (not fold-safe; CV below is fold-safe)')
plt.tight_layout()
plt.savefig('../data/processed/fig_rh_residual_vs_lagged_pm25.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fold-wise residual + Ridge CV (no leakage) ---
N_PM = 1
N_CTRL_NT = len(CONTROL_COLS_NO_T)
ctrl_nt_idx = list(range(N_PM, N_PM + N_CTRL_NT))

pre_m1_nt = ColumnTransformer(
    [
        ('pm_pass', 'passthrough', [0]),
        ('ctrl_scale', StandardScaler(), ctrl_nt_idx),
    ]
)
pre_m0_nt = ColumnTransformer(
    [('ctrl_scale', StandardScaler(), list(range(N_CTRL_NT)))]
)


def cv_ridge_on_rh_residual_foldwise(
    X_pm, X_ctrl_no_t, T, RH, n_splits=5
):
    """Remove linear RH~T per fold (train-only fit), then Ridge M0/M1 on residual."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(tscv.split(T)):
        lr = LinearRegression()
        lr.fit(T[tr].reshape(-1, 1), RH[tr])
        res_tr = RH[tr] - lr.predict(T[tr].reshape(-1, 1))
        res_te = RH[te] - lr.predict(T[te].reshape(-1, 1))

        X0_tr = X_ctrl_no_t[tr]
        X0_te = X_ctrl_no_t[te]
        X1_tr = np.column_stack([X_pm[tr], X_ctrl_no_t[tr]])
        X1_te = np.column_stack([X_pm[te], X_ctrl_no_t[te]])

        m0 = Pipeline(
            [('prep', pre_m0_nt), ('ridge', RidgeCV(alphas=alphas, cv=3))]
        )
        m1 = Pipeline(
            [('prep', pre_m1_nt), ('ridge', RidgeCV(alphas=alphas, cv=3))]
        )

        m0.fit(X0_tr, res_tr)
        m1.fit(X1_tr, res_tr)

        for name, model, Xtr, Xev, ytr, yev in [
            ('M0_controls', m0, X0_tr, X0_te, res_tr, res_te),
            ('M1_ctrl+PMlag', m1, X1_tr, X1_te, res_tr, res_te),
        ]:
            pred_tr = model.predict(Xtr)
            pred_te = model.predict(Xev)
            rows.append(
                {
                    'fold': fold,
                    'model': name,
                    'train_rmse': np.sqrt(mean_squared_error(ytr, pred_tr)),
                    'test_rmse': np.sqrt(mean_squared_error(yev, pred_te)),
                    'train_r2': r2_score(ytr, pred_tr),
                    'test_r2': r2_score(yev, pred_te),
                }
            )
    return pd.DataFrame(rows)


X_pm2 = work[PM_LAG_COL].values.reshape(-1, 1)
X_ctrl_nt = work[CONTROL_COLS_NO_T].values
T_arr = work['temperature_f'].values
RH_arr = work['humidity'].values

res_cv = cv_ridge_on_rh_residual_foldwise(X_pm2, X_ctrl_nt, T_arr, RH_arr, n_splits=5)
agg_r = res_cv.groupby('model')[['train_rmse', 'test_rmse', 'train_r2', 'test_r2']].mean()
d_r = agg_r.loc['M1_ctrl+PMlag', 'test_rmse'] - agg_r.loc['M0_controls', 'test_rmse']

print('=' * 60)
print('Outcome: RH residual (linear T removed, fold-wise — no leakage)')
print('=' * 60)
print(agg_r.round(4).to_string())
print(
    f"\nMean test RMSE(M1) - Mean test RMSE(M0) = {d_r:+.4f}  (negative => M1 better)"
)

# Full-sample coef on PM for residual (after global RH~T for interpretability only)
res_all = work['humidity'].values - lr_eda.predict(work[['temperature_f']])
X1_nt = np.column_stack([work[PM_LAG_COL].values, work[CONTROL_COLS_NO_T].values])
pipe_r = Pipeline(
    [
        ('prep', pre_m1_nt),
        ('ridge', RidgeCV(alphas=alphas, cv=TimeSeriesSplit(n_splits=5))),
    ]
)
pipe_r.fit(X1_nt, res_all)
beta_pm_res = pipe_r.named_steps['ridge'].coef_[0]
print(
    f"\nExploratory full-sample β_PM on RH residual (after global linear T): {beta_pm_res:+.6f} per µg/m³"
)

## 3. Time-series CV: does lagged PM2.5 improve prediction of moisture?

**M0:** Ridge on **scaled** controls only.  
**M1:** Ridge on **scaled** controls + **raw** lagged PM2.5 (first column passes through unscaled so the coefficient is in **µg/m³** units if we only scale controls — here we scale all *except* PM in a `ColumnTransformer`).

We report **mean test RMSE** and **mean test R²** across `TimeSeriesSplit` folds. **Lower RMSE** for M1 suggests the lag adds predictive information for that outcome under this linear specification.

In [ ]:
N_PM = 1
N_CTRL = len(CONTROL_COLS)
ctrl_idx = list(range(N_PM, N_PM + N_CTRL))

pre_m1 = ColumnTransformer(
    [
        ('pm_pass', 'passthrough', [0]),
        ('ctrl_scale', StandardScaler(), ctrl_idx),
    ]
)

pre_m0 = ColumnTransformer(
    [('ctrl_scale', StandardScaler(), list(range(N_CTRL)))],
)

alphas = np.logspace(-2, 4, 40)


def cv_ridge_m0m1(X_pm, X_ctrl, y, n_splits=5, random_state=0):
    """Return dict of mean train/test RMSE and R2 for M0 and M1."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(tscv.split(X_ctrl)):
        X1_tr = np.column_stack([X_pm[tr], X_ctrl[tr]])
        X1_te = np.column_stack([X_pm[te], X_ctrl[te]])
        X0_tr, X0_te = X_ctrl[tr], X_ctrl[te]

        m0 = Pipeline(
            [('prep', pre_m0), ('ridge', RidgeCV(alphas=alphas, cv=3))]
        )
        m1 = Pipeline(
            [('prep', pre_m1), ('ridge', RidgeCV(alphas=alphas, cv=3))]
        )

        m0.fit(X0_tr, y[tr])
        m1.fit(X1_tr, y[tr])

        for name, model, Xtr, Xev in [
            ('M0_controls', m0, X0_tr, X0_te),
            ('M1_ctrl+PMlag', m1, X1_tr, X1_te),
        ]:
            pred_tr = model.predict(Xtr)
            pred_te = model.predict(Xev)
            rows.append(
                {
                    'fold': fold,
                    'model': name,
                    'train_rmse': np.sqrt(mean_squared_error(y[tr], pred_tr)),
                    'test_rmse': np.sqrt(mean_squared_error(y[te], pred_te)),
                    'train_r2': r2_score(y[tr], pred_tr),
                    'test_r2': r2_score(y[te], pred_te),
                }
            )
    return pd.DataFrame(rows)


X_pm = work[PM_LAG_COL].values.reshape(-1, 1)
X_ctrl = work[CONTROL_COLS].values

summary_rows = []
for out_col, out_label in OUTCOMES.items():
    y = work[out_col].values
    res = cv_ridge_m0m1(X_pm, X_ctrl, y, n_splits=5)
    agg = res.groupby('model')[['train_rmse', 'test_rmse', 'train_r2', 'test_r2']].mean()
    print('\n' + '=' * 60)
    print(out_label)
    print('=' * 60)
    print(agg.round(4))
    d_rmse = agg.loc['M1_ctrl+PMlag', 'test_rmse'] - agg.loc['M0_controls', 'test_rmse']
    print(f"\nMean test RMSE(M1) - Mean test RMSE(M0) = {d_rmse:+.4f}  (negative => M1 better)")
    summary_rows.append(
        {
            'outcome': out_col,
            'delta_test_rmse': d_rmse,
            'M0_mean_test_rmse': agg.loc['M0_controls', 'test_rmse'],
            'M1_mean_test_rmse': agg.loc['M1_ctrl+PMlag', 'test_rmse'],
            'M0_mean_test_r2': agg.loc['M0_controls', 'test_r2'],
            'M1_mean_test_r2': agg.loc['M1_ctrl+PMlag', 'test_r2'],
        }
    )

cv_summary = pd.DataFrame(summary_rows)
cv_summary

## 4. Coefficient on lagged PM2.5 (full-sample Ridge, same alphas)

After fitting on **all** rows (exploratory — use CV above for honest performance), the **coefficient on lagged PM2.5** is in **outcome units per µg/m³** because that column is not standardized.

In [ ]:
X1 = np.column_stack([work[PM_LAG_COL].values, work[CONTROL_COLS].values])

rows = []
for out_col, out_label in OUTCOMES.items():
    y = work[out_col].values
    pipe = Pipeline(
        [('prep', pre_m1), ('ridge', RidgeCV(alphas=alphas, cv=TimeSeriesSplit(n_splits=5)))]
    )
    pipe.fit(X1, y)
    beta_pm = pipe.named_steps['ridge'].coef_[0]
    alpha = pipe.named_steps['ridge'].alpha_
    rows.append({'outcome': out_col, 'beta_pm_per_ugm3': beta_pm, 'ridge_alpha': alpha})
    print(f"{out_col:22s}  β_PM = {beta_pm:+.5f}  (per µg/m³)   α = {alpha:.4f}")

coef_tbl = pd.DataFrame(rows)
coef_tbl

## 5. How to read this for the CCN angle

- If **M1** lowers **mean test RMSE** for **humidity** or **dewpoint depression** in a **consistent** way across folds, you can report that **lagged smoke-era PM2.5 carries information about moisture not already captured** by T, p, wind, and calendar — **weakly consistent** with pathways that alter the moist layer (including but not limited to CCN effects).  
- If **M1 never helps** (or **β_PM** is tiny and CV favors M0), the honest story is: **in this station, season, and hour resolution, a linear lag effect is not detectable** above strong thermodynamic controls — that does **not** disprove CCN physics globally; it bounds what **your** data can say.  
- **Visibility** mixes smoke aerosol with fog/rain; use it only as a **secondary**, clearly labeled ambiguity.

- **Section 2b** removes **linear** `RH ~ T` **inside each CV fold** (no leakage), then repeats M0 vs M1 on the residual with **p, wind, hour, doy** as controls. Compare that table to **section 3** (raw RH with T still in the control set).

**Next extensions (if you continue):** quadratic `RH ~ T + T²` in the first stage, **rolling 24h RH anomaly**, or an event study with **forward** precipitation (watch leakage).